# Community registry

Install a semantic-model package from a local directory (or later, from a tarball / URL / hosted registry name). Once installed, the retriever surfaces its definitions alongside your own YAML semantic models.

Here we install the bundled `stripe-semantic` package and verify its metrics and examples are loaded. We don't have a Stripe schema wired up, so we don't ask Stripe-specific questions end-to-end — but the package format and retrieval integration are the interesting parts.

In [1]:
import os
from pathlib import Path

# Locate project root (so relative paths data/, packages/, .env resolve regardless of
# where the notebook was launched from).
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('could not locate project root (no pyproject.toml found upward)')
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv()
if not (os.environ.get('OPENAI_API_KEY') or os.environ.get('ANTHROPIC_API_KEY')):
    raise RuntimeError('Set OPENAI_API_KEY or ANTHROPIC_API_KEY in your .env before running this notebook.')
print('project root:', PROJECT_ROOT)

project root: /Users/nitingupta/Desktop/talkdb


In [2]:
from talkdb.config.settings import get_settings
from talkdb.core.engine import Engine

engine = Engine(get_settings())
print('engine ready')

engine ready


## Install the bundled `stripe-semantic` package

Idempotent — reinstalls on top of any existing copy.

In [3]:
result = engine.install_package('./packages/stripe-semantic')
for k, v in result.items():
    print(f'  {k}: {v}')

  installed: True
  name: stripe-semantic
  version: 1.0.0
  source_path: /Users/nitingupta/Desktop/talkdb/data/packages/stripe-semantic
  example_count: 6


## List installed packages

In [4]:
for p in engine.list_installed_packages():
    print(f"  {p['name']}@{p['version']} — {p['example_count']} examples, schema_type={p['schema_type']}")

  stripe-semantic@1.0.0 — 6 examples, schema_type=stripe


## Search the registry

In [5]:
for hit in engine.search_registry('stripe'):
    print(f"  [{'✓' if hit['verified'] else ' '}] {hit['name']}@{hit['version']} ({hit['schema_type']})")
    if hit.get('description'):
        print(f"      {hit['description']}")

remote search failed, falling back to local: [Errno 8] nodename nor servname provided, or not known


  [ ] stripe-semantic@1.0.0 (stripe)
      Semantic model for Stripe's core tables: charges, customers, subscriptions, invoices. Covers MRR, churn, and LTV metric definitions.


## Inspect what the package actually contains

Every package is pure YAML — no executable code, by design. Let's load it and look at its metric definitions and example queries.

In [6]:
from talkdb.registry.package import SemanticPackage

pkg = SemanticPackage.load('./packages/stripe-semantic')
print(f'{pkg.manifest.name}@{pkg.manifest.version} — {pkg.manifest.description}')
print(f'{len(pkg.semantic_model.metrics)} metrics, {len(pkg.semantic_model.tables)} tables, {len(pkg.all_examples)} examples')
print()
print('Metrics:')
for m in pkg.semantic_model.metrics:
    print(f'  • {m.name} — {m.description}')
    print(f'      {m.calculation}')

stripe-semantic@1.0.0 — Semantic model for Stripe's core tables: charges, customers, subscriptions, invoices. Covers MRR, churn, and LTV metric definitions.
5 metrics, 5 tables, 6 examples

Metrics:
  • mrr — Monthly Recurring Revenue. Sum of active subscription plan amounts, converted from cents to dollars.
      SUM(subscriptions.plan_amount) / 100.0 WHERE subscriptions.status = 'active'
  • active_subscriptions — Count of subscriptions currently in active status.
      COUNT(*) WHERE subscriptions.status = 'active'
  • gross_charges — Sum of successful charges in dollars (amount is stored in cents).
      SUM(charges.amount) / 100.0 WHERE charges.status = 'succeeded'
  • net_revenue — Gross charges minus refunds, in dollars.
      (SUM(charges.amount) - COALESCE(SUM(refunds.amount), 0)) / 100.0 WHERE charges.status = 'succeeded'
  • customer_ltv — Lifetime value: total successful charges per customer, in dollars.
      SUM(charges.amount) / 100.0 WHERE charges.status = 'succeeded' G

## Example queries shipped with the package

In [7]:
for ex in pkg.all_examples[:4]:
    print(f"Q: {ex.question}")
    print(f"A: {ex.sql}")
    print()

Q: What is our MRR?
A: SELECT SUM(plan_amount) / 100.0 AS mrr FROM subscriptions WHERE status = 'active'

Q: How many active subscriptions do we have?
A: SELECT COUNT(*) AS n FROM subscriptions WHERE status = 'active'

Q: What is our gross revenue this month?
A: SELECT SUM(amount) / 100.0 AS gross_revenue FROM charges WHERE status = 'succeeded' AND created >= strftime('%s', 'now', 'start of month')

Q: MRR by subscription plan
A: SELECT plan_id, SUM(plan_amount) / 100.0 AS mrr FROM subscriptions WHERE status = 'active' GROUP BY plan_id



## Uninstall when done

In [8]:
result = engine.uninstall_package('stripe-semantic')
print(result)

{'removed': True, 'name': 'stripe-semantic'}
